In [ ]:
import matplotlib.pylab as plt
import matplotlib as mpl
import xarray as xr
import pint_xarray
import numpy as np
import cftime
from functools import partial

from pism_terra.processing import integrate_rate, preprocess_netcdf, normalize_timeseries

ref_year = "2007"

In [ ]:
ds_free = xr.open_dataset("/Users/andy/base/pism-terra/2026_08_ismip7_hist_1985_free//output/processed_scalar//basin_g900m_id_CESM2-WACCM_historical_free_1985-01-01_2015-01-01.nc")
ds_free = ds_free.expand_dims({"uq_id": ["free"]})
free_dx = ds_free.pism_config.attrs["grid.dx"]
free_dy = ds_free.pism_config.attrs["grid.dy"]
ds_free["grounding_line_flux_nonneg"] = ds_free.grounding_line_flux_nonneg.pint.quantify() * xr.DataArray(free_dx).pint.quantify("m") * xr.DataArray(free_dy).pint.quantify("m")
ds_free["grounding_line_flux"] = ds_free.grounding_line_flux.pint.quantify() * xr.DataArray(free_dx).pint.quantify("m") * xr.DataArray(free_dy).pint.quantify("m")

ds_prescribed = xr.open_dataset("/Users/andy/base/pism-terra/2026_08_ismip7_hist_1985_prescribed/output/processed_scalar/basin_g900m_id_CESM2-WACCM_historical_free_1985-01-01_2015-01-01.nc")
ds_prescribed = ds_prescribed.expand_dims({"uq_id": ["prescribed"]})
prescribed_dx = ds_prescribed.pism_config.attrs["grid.dx"]
prescribed_dy = ds_free.pism_config.attrs["grid.dy"]
ds_prescribed["grounding_line_flux_nonneg"] = ds_prescribed.grounding_line_flux_nonneg.pint.quantify() * xr.DataArray(prescribed_dx).pint.quantify("m") * xr.DataArray(prescribed_dy).pint.quantify("m")
ds_prescribed["grounding_line_flux"] = ds_prescribed.grounding_line_flux.pint.quantify() * xr.DataArray(prescribed_dx).pint.quantify("m") * xr.DataArray(prescribed_dy).pint.quantify("m")

# ds_prescribed_long = xr.open_dataset("/Users/andy/base/pism-terra/2026_08_ismip7_hist_long_prescribed/output/processed_scalar/basin_g3600m_id_CESM2-WACCM_historical_prescribed_1985-01-01_2015-01-01.nc")
# ds_prescribed_long = ds_prescribed_long.expand_dims({"uq_id": ["prescribed_long"]})
# prescribed_long_dx = ds_prescribed_long.pism_config.attrs["grid.dx"]
# prescribed_long_dy = ds_free.pism_config.attrs["grid.dy"]
# ds_prescribed_long["grounding_line_flux_nonneg"] = ds_prescribed_long.grounding_line_flux_nonneg.pint.quantify() * xr.DataArray(prescribed_long_dx).pint.quantify("m") * xr.DataArray(prescribed_long_dy).pint.quantify("m")
# ds_prescribed_long["grounding_line_flux"] = ds_prescribed_long.grounding_line_flux.pint.quantify() * xr.DataArray(prescribed_long_dx).pint.quantify("m") * xr.DataArray(prescribed_long_dy).pint.quantify("m")


ds_inv = xr.open_dataset("/Users/andy/base/pism-terra/2026_08_ismip7_hist_2007_inv_prescribed/output/processed_scalar/basin_g1500m_id_CESM2-WACCM_historical_prescribed_2007-01-01_2015-01-01.nc")
ds_inv = ds_inv.expand_dims({"uq_id": ["inv"]})
inv_dx = ds_inv.pism_config.attrs["grid.dx"]
inv_dy = ds_inv.pism_config.attrs["grid.dy"]
ds_inv["grounding_line_flux_nonneg"] = ds_inv.grounding_line_flux_nonneg.pint.quantify() * xr.DataArray(inv_dx).pint.quantify("m") * xr.DataArray(inv_dy).pint.quantify("m")
ds_inv["grounding_line_flux"] = ds_inv.grounding_line_flux.pint.quantify() * xr.DataArray(inv_dx).pint.quantify("m") * xr.DataArray(inv_dy).pint.quantify("m")

ds = xr.concat([ds_free, ds_prescribed, ds_inv], compat="no_conflicts", join="outer", dim="uq_id", data_vars="all")
#ds = xr.concat([ds_free, ds_prescribed], compat="no_conflicts", join="outer", dim="uq_id", data_vars="all")

ds = ds.set_index(glacier_id="glacier_id_name")
ds = ds.assign_coords(glacier_idn=np.char.add("GIS_", ds.glacier_id.values))
basins = ds.glacier_id


In [ ]:
ds = ds.convert_calendar("standard", use_cftime=False).resample(time='MS').mean('time').pint.quantify()

In [ ]:
grace = xr.open_dataset("/Users/andy/base/pism-ragis/data/grace/greenland_mass_balance.nc").squeeze().pint.quantify()
mankoff = xr.open_dataset("/Users/andy/base/pism-ragis/data/mass_balance/mankoff_greenland_mass_balance_clean.nc").drop_vars(["MB_HIRHAM", "MB_MAR", "MB_RACMO"]).pint.quantify()
mankoff = mankoff.pint.to("Gt/yr").sel({"time": slice("1985", "2015")})
mankoff = mankoff.assign_coords(region=np.char.add("GIS_", mankoff.region.values))
mankoff_gis = mankoff[["MB", "MB_err", "SMB", "SMB_err", "D", "D_err", "BMB", "BMB_err"]].rename_vars({"MB": "MB_ROI", "MB_err": "MB_ROI_err", 
                                                                                     "SMB": "SMB_ROI", "SMB_err": "SMB_ROI_err",
                                                                                    "D": "D_ROI", "D_err": "D_ROI_err",
                                                                                    "BMB": "BMB_ROI", "BMB_err": "BMB_ROI_err"}).expand_dims({"region": ["GIS_GIS"]})

mankoff = xr.concat([mankoff.drop_vars(["MB", "MB_err", "SMB", "SMB_err", "D", "D_err", "BMB", "BMB_err"]), mankoff_gis], dim="region")
sigma = 2.0
mankoff_cmb = integrate_rate(mankoff.MB_ROI)
mankoff_cmb = (mankoff_cmb - mankoff_cmb.sel(time=ref_year, method="nearest"))
mankoff_mb = mankoff.MB_ROI.resample(time='YS').mean("time")
mankoff_mb_err = mankoff.MB_ROI_err.resample(time='YS').mean("time")
mankoff_smb = mankoff.SMB_ROI.resample(time='YS').mean("time")
mankoff_smb_err = mankoff.SMB_ROI_err.resample(time='YS').mean("time")
mankoff_glf = -mankoff.D_ROI.resample(time='YS').mean("time")
mankoff_glf_err = -mankoff.D_ROI_err.resample(time='YS').mean("time")



In [ ]:
mankoff

In [ ]:
glf = ds.grounding_line_flux.pint.to("Gt/yr")
smb = ds.tendency_of_ice_mass_due_to_surface_mass_flux
mb = ds.tendency_of_ice_mass
mb = smb + glf

mass = integrate_rate(mb)

mass = mass - mass.sel(time=ref_year, method="nearest")
mass = mass.pint.to("Gt")

for basin in basins:
    _mass = mass.sel(glacier_id=basin)
    _mankoff_cmb = mankoff_cmb.sel(region=basin)
    fig, ax = plt.subplots(1, 1, figsize=(6.4, 4.8))
    _mass.plot(hue="uq_id", ax=ax, lw=1, add_legend=True)
    ax.set_prop_cycle(None)
    _mankoff_cmb.plot(ax=ax, lw=1.5, color="0.5")
    ax.set_xlim(np.datetime64("1985"), np.datetime64("2015"))
    fig.tight_layout()
    fig.savefig(f"gris_{basin.values}_cmb.png", dpi=300)

In [ ]:
fig, ax = plt.subplots(1, 1)
mass.plot(hue="uq_id", ax=ax, lw=1, add_legend=True)
ax.set_prop_cycle(None)
mankoff_cmb.plot(ax=ax, lw=2, color="0.5")
ax.set_xlim(np.datetime64("1985"), np.datetime64("2000"))

In [ ]:

rc_params = {
    "font.size": 6,
        # Add other rcParams settings if needed
}

with mpl.rc_context(rc=rc_params):

    for basin in basins:
        _mankoff_mb = mankoff_mb.sel(region=basin)
        _mankoff_mb_err = mankoff_mb_err.sel(region=basin)
        _mankoff_smb = mankoff_smb.sel(region=basin)
        _mankoff_smb_err = mankoff_smb_err.sel(region=basin)
        _mankoff_glf = mankoff_glf.sel(region=basin)
        _mankoff_glf_err = mankoff_glf_err.sel(region=basin)
        _mb = mb.sel(glacier_id=basin)
        _smb = smb.sel(glacier_id=basin)
        _glf = glf.sel(glacier_id=basin)
    
        fig, axs = plt.subplots(3, 1, sharex=True, figsize=(4.8, 4.8))
        axs[0].fill_between(_mankoff_mb.time, _mankoff_mb - sigma * _mankoff_mb_err, _mankoff_mb + sigma * _mankoff_mb_err, color="0.5", alpha=0.25, lw=0)
        axs[1].fill_between(_mankoff_mb.time, _mankoff_smb - sigma * _mankoff_smb_err, _mankoff_smb + sigma * _mankoff_smb_err, color="0.5", alpha=0.25, lw=0)
        axs[2].fill_between(_mankoff_mb.time, _mankoff_glf - sigma * _mankoff_glf_err, _mankoff_glf + sigma * _mankoff_glf_err, color="0.5", alpha=0.25, lw=0)
        
        _mankoff_mb.resample(time='YS').mean('time').plot(ax=axs[0], color="0.5", lw=2)
        _mankoff_smb.resample(time='YS').mean('time').plot(ax=axs[1], color="0.5", lw=2)
        _mankoff_glf.resample(time='YS').mean('time').plot(ax=axs[2], color="0.5", lw=2)
        for k, (da, ls) in enumerate([(_mb, "solid"), (_smb, "dotted"), (_glf, "dashed")]):
            ax = axs[k]
            da.compute().resample(time='YS').mean('time').plot(hue="uq_id", ax=ax, lw=1, add_legend=False if k > 0 else True)        
            ax.set_title(None)
            ax.set_xlabel(None)
            ax.axhline(0, color="k", lw=0.5, ls="dotted")
            ax.set_xlim(np.datetime64("1985"), np.datetime64("2015"))
            #ax.set_ylim(-1000, 1000)
        axs[-1].set_xlabel("Year")
        fig.tight_layout()
        fig.savefig(f"fluxes_{basin.values}.pdf")


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6.4, 6.2))
ax.fill_between(mankoff_mb.time, mankoff_mb - sigma * mankoff_mb_err, mankoff_mb + sigma * mankoff_mb_err, color="0.5", alpha=0.25, lw=0)
ax.fill_between(mankoff_mb.time, mankoff_smb - sigma * mankoff_smb_err, mankoff_smb + sigma * mankoff_smb_err, color="0.5", alpha=0.25, lw=0)
ax.fill_between(mankoff_mb.time, mankoff_glf - sigma * mankoff_glf_err, mankoff_glf + sigma * mankoff_glf_err, color="0.5", alpha=0.25, lw=0)

mankoff_mb.resample(time='YS').mean('time').plot(ax=ax, color="0.5", lw=2, ls="solid")
mankoff_smb.resample(time='YS').mean('time').plot(ax=ax, color="0.5", lw=2, ls="dashed")
mankoff_glf.resample(time='YS').mean('time').plot(ax=ax, color="0.5", lw=2, ls="dotted")

mb.compute().resample(time='YS').mean('time').plot(hue="uq_id", ax=ax, lw=1, ls="solid", add_legend=True)        
ax.set_prop_cycle(None)
smb.compute().resample(time='YS').mean('time').plot(hue="uq_id", ax=ax, lw=1, ls="dashed", add_legend=False)        
ax.set_prop_cycle(None)
glf.compute().resample(time='YS').mean('time').plot(hue="uq_id", ax=ax, lw=1, ls="dotted", add_legend=False)     
ax.set_prop_cycle(None)


ax.set_title(None)
ax.axhline(0, color="k", lw=0.5, ls="dotted")
ax.set_xlim(np.datetime64("1985"), np.datetime64("2015"))
ax.set_ylim(-800, 800)
fig.savefig("fluxes.pdf")


In [ ]:
import xarray as xr
from pism_terra.processing import preprocess_netcdf
from functools import partial

pre = partial(preprocess_netcdf,
              exp_regexp=(r"_id_(.+?)_uq_\d+_", r"_id_(.+?)_\d{4}-\d{2}-\d{2}", r"id_(.+?)_"))

ds = xr.open_mfdataset(
    "/mnt/storstrommen/pism/terra/2026_08_ismip7_core_test/output/scalar/scalar_g3600m_id_*.nc",
    preprocess=pre, combine="nested", concat_dim="exp_id", join="outer", decode_timedelta=True,
)

xr.open_mfdataset("/mnt/storstrommen/pism/terra/2026_08_ismip7_core_test/output/GrIS/UAF/PISM/CORE/C0*/tend*.nc")


In [ ]:
ice_mass_glacierized = ds.ice_mass_glacierized - ds.ice_mass_glacierized.sel(time="2015", method="nearest")
ice_mass_glacierized.plot(hue="exp_id")

In [ ]:
import glob, os
import xarray as xr
from functools import partial
import pint_xarray
from pism_terra.processing import preprocess_netcdf

ROOT = "/mnt/storstrommen/pism/terra/2026_08_ismip7_core_test/output/GrIS/UAF/PISM/CORE"
pre = partial(preprocess_netcdf, exp_regexp=r"/CORE/(C\d+)/", process_config=False, drop_vars=["time_bounds"], drop_dims=["nv"])

# Group by the C* directory — that is what exp_id has to come from.
groups = {}
for f in sorted(glob.glob(f"{ROOT}/C*/tend*.nc")):
    groups.setdefault(os.path.basename(os.path.dirname(f)), []).append(f)

ds = xr.open_mfdataset(
    [groups[k] for k in sorted(groups)],
    preprocess=pre, combine="nested", concat_dim=["exp_id", None],
    join="outer", compat="no_conflicts", decode_timedelta=True,
)

In [ ]:
ds.tendligroundf.plot(hue="exp_id")

In [ ]:
ds.tendacabf.pint.quantify()